# Week 2 Day 4

## Covered Today
1. How LLM Tool Calling Really Works (No Magic, Just Prompts)
2. Common Use Cases for LLM Tools and Agentic AI Workflows
3. Building an Airline AI Assistant with Tool Calling in OpenAI and Gradio
4. Handling Multiple Tool Calls with OpenAI and Gradio
5. Building Tool Calling with SQLite Database Integration

In [5]:
# we start by setting up the entire boilerplate for AI LLM interacion


# required imports
import os
from dotenv import load_dotenv
import requests
from openai import OpenAI
from IPython.display import Markdown, display, update_display
import gradio as gr 
import json


# load API Keys
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')


# check if all loaded keys exist and are as per the format
if openai_api_key:
    if openai_api_key.startswith("sk-"):
        print(f"OpenAI      : OK           (begins {openai_api_key[:7]}...)")
    else:
        print("OpenAI      : WRONG FORMAT (should start with 'sk-')")
else:
    print("OpenAI      : MISSING")

if anthropic_api_key:
    if anthropic_api_key.startswith("sk-ant-"):
        print(f"Anthropic   : OK           (begins {anthropic_api_key[:10]}...)")
    else:
        print("Anthropic   : WRONG FORMAT (should start with 'sk-ant-')")
else:
    print("Anthropic   : MISSING")

if google_api_key:
    if google_api_key.startswith("AQ.Ab"):
        print(f"Google      : OK           (begins {google_api_key[:5]}...)")
    else:
        print("Google      : WRONG FORMAT (should start with 'AQ.Ab' or 'AIza')")
else:
    print("Google      : MISSING")

if openrouter_api_key:
    if openrouter_api_key.startswith("sk-or-"):
        print(f"OpenRouter  : OK           (begins {openrouter_api_key[:8]}...)")
    else:
        print("OpenRouter  : WRONG FORMAT (should start with 'sk-or-')")
else:
    print("OpenRouter  : MISSING")


# create clients for each provider
openai_client = OpenAI()
google_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'
anthropic_url = 'https://api.anthropic.com/v1/'
openrouter_url = 'https://openrouter.ai/api/v1'
ollama_url = 'http://127.0.0.1:11434/v1'


google_client = OpenAI(base_url=google_url, api_key=google_api_key)
anthropic_client = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)
openrouter_client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama_client = OpenAI(base_url=ollama_url, api_key='Ollama')

OpenAI      : OK           (begins sk-proj...)
Anthropic   : OK           (begins sk-ant-api...)
Google      : OK           (begins AQ.Ab...)
OpenRouter  : OK           (begins sk-or-v1...)


In [6]:
# Now we move ahead with setting up our airline assistant
system_message = "You are a helpful assistant for an airline called flighty. Give short, courteous answers, no more than 1 sentence. Always be accurate. If you are not aware of any information, politely inform the user so."

In [7]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    response = openai_client.chat.completions.create(model='gpt-5-mini', messages=messages) #type: ignore
    
    return response.choices[0].message.content


In [8]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [9]:
ticket_prices = {"london": "$1999", "madrid": "$999", "chennai": "$199", "mumbai": "$299", "toronto": "$1499"}

In [10]:
def get_ticket_price(destination_city):
    print(f"Tool called for city: {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of the ticket to {destination_city} is {price}."

In [11]:
get_ticket_price("london")
# so far, we have just created a dict with cities and their prices and then we have created a function to get the price from the list. Now, in order to pass on this information to the LLM, we use a fixed json format, where we describe this function

Tool called for city: london


'The price of the ticket to london is $1999.'

In [12]:
# Now, we can not provide the above function to the LLM directly, we need to pass on the information of the function in json format. For this, we save the json in a variable. 

# Let's start by declaring the variable and giving it an empty dict (format used for json)
price_function = {}

# next, we add the name of the function, which will be acting as a tool.
price_function = {
    "name": "get_ticket_price"
}

In [13]:
# Once the name is passed, we move on to pass on the description of the function, here, we have to be completely unambigous, since, this description will actually tell the LLM, what this function does. In our case, this function simply gives the price of the return ticket to a destiation city. So, we add this information in the function.
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city"
}

In [14]:
# Now, a function also contains parameters and to make this function run, we would need arguments to replace the parameters while running these functions, so for that we would need to also explain these parameters to the function. Now, parameter information in this json is being passed on as a dict, which is also called Object in json language. So, we simply add, parameters as a key and in this open a new dict and pass on the type as object. This also tells the LLM that you should also, return the information in the form of the object. As a reply to this tool call, the LLM would return {"destination_city": "london"}

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city",
    "parameters": {
        "type": "object"
    }
}

In [15]:
# Now to explain each parameter to be passed on in this function, we add a new key, properties, which can be used to cover all the parameters of this function. Now, once we add the properties key, we add a new dict to it, which will contain the information of each paramter.

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city",
    "parameters": {
        "type": "object",
        "properties": {
            
        }
    }
}

In [16]:
# now we pass on each paramter one by one, describe its type and give its description.

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The destination city, or the city that the customer wants to travel to."
            }
        }
    }
}

# under properties, the way we have added destination_city, we can add more parameters if the function has more parameters. for example, if the function also takes in num_pax(number of passengers, then we can pass on this also as below)
"""
"parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The destination city, or the city that the customer wants to travel to."
            },
            "num_pax": {
                "type": "integer",
                "description": "Total count of passengers travelling to this destination city."
            }
        }
    }
"""
# since, our function only wants one parameter, we keep it as is

'\n"parameters": {\n        "type": "object",\n        "properties": {\n            "destination_city": {\n                "type": "string",\n                "description": "The destination city, or the city that the customer wants to travel to."\n            },\n            "num_pax": {\n                "type": "integer",\n                "description": "Total count of passengers travelling to this destination city."\n            }\n        }\n    }\n'

In [17]:
# once all the parameters are defined, we inform the LLM if there is any paramters which is required in order to run the function. in our case, it is destination_city. we add a new "required" key iun the parameters and pass on a list of required parameters.

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The destination city, or the city that the customer wants to travel to."
            }
        },
        "required": ["destination_city"]
    }
}

In [18]:
# the LLM is as we know a token predicter, hence, it can also pass in information back that is not needed. for example, if the user tells it is for 2 passengers, but we are not passing this information to the function. If that additional information comes across, we we will get an error while running the function. so, we add another key in parameters by the name of additionalProperties (camel case) and mark its value as False. There are two more values that can be passed, but we will stick to false for now. 

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The destination city, or the city that the customer wants to travel to."
            }
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

# this marks the completion of this json

In [19]:
# now, this is added to a list. This list can be used to pass one or more such descriptions. Since, we can have multiple tools, hence, we define it separately.

tools = [{"type": "function", "function": price_function}]

In [20]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The destination city, or the city that the customer wants to travel to.'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [21]:
# # Now in order to make this work, with the LLM, we need to first pass on these details to the LLM, basically the details of the functions or tools. Then, if the LLM decides that it wants to run the tool, we then run the tool and pass on the details again to the LLM, so that it can add the info it received from the function and pass it on to the users. So, this would require us to call the LLM twice if we are using a tool. 

# we start by passing in the information that these tools exist to the LLM. we do this by simply passing in info of the tools in the create() function as below.

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai_client.chat.completions.create(model='gpt-4.1-mini', messages=messages, tools=tools) #type: ignore

    # now, to see if LLM is asking for a tool call, we do not print the content, we stop at response
    return response


In [22]:
# let's try running this function, to understand what models replies back with if asked for a ticket price

chat('looking to book a ticket to london', [])

ChatCompletion(id='chatcmpl-E9pv8UU3T3RByKfywupb1vwJG4MPv', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_fCDNSgESWB6Oy5W5yhR5Bjf3', function=Function(arguments='{"destination_city":"london"}', name='get_ticket_price'), type='function')]))], created=1786013242, model='gpt-4.1-mini-2025-04-14', object='chat.completion', moderation=None, service_tier='priority', system_fingerprint='fp_60b7892dc5', usage=CompletionUsage(completion_tokens=17, prompt_tokens=118, total_tokens=135, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0)))

In [23]:
# We see in the last response, when we simply asked for the price of the ticket to london, we now look at the entire response. 

# first thing to notice is that under choices we have:
# finish_reason='tool_calls' 
# this means, that the LLM stopped since it needs to call a tool to proceed further

# Now, another thing to notice here is the information we have received:
# tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_baN14cQ1Dkfb8ejGwtjVxfGH', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function')])

# in tool calls we have received an ID: id='call_baN14cQ1Dkfb8ejGwtjVxfGH'
# then, we have function details: function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function')
# which contain: 
# arguments='{"destination_city":"London"}' object format
# name of the function: name='get_ticket_price'
# type of the function: type='function'


# now, we create a function, with our understanding, that takes the input from the LLM for the tool call, calls the tool and returns a LLM understandable response. We create a parameter message, that we shall pass in this function.

# the parameter that we are passing is: message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_7xZ8Jd7IayC6bHrCvsPN0eUm', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function')]))

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    # the above will give us: ChatCompletionMessageFunctionToolCall(id='call_7xZ8Jd7IayC6bHrCvsPN0eUm', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function')
    # now, from this, we can extract the function name by using tool_call.function.name, so we do
    if tool_call.function.name == 'get_ticket_price':
        # this will only run if the function name is above, then we know that there would be an adjacent city name also, which is here: {"destination_city":"London"}
        # since this is in json, we do
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get("destination_city")
        # now, since we have the city, we can run the given function to get the price, we do
        price_details = get_ticket_price(city)
        # and now, we return this information in LLM understandable format, which is
        response = {
            'role': 'tool',
            'content': price_details,
            "tool_call_id": tool_call.id
        }

    return response

In [24]:
# now that we know this, we advance from here and improve our chat function to run a second call.



def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai_client.chat.completions.create(model='gpt-4.1-mini', messages=messages, tools=tools) #type: ignore

    # we now understand that if LLM wants to run a tool, then it gives finish_reason='tool_calls', we can use this to run the tool
    if response.choices[0].finish_reason == 'tool_calls':

        # we now extract the message that we have received from the LLM, which will be
        # message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_7xZ8Jd7IayC6bHrCvsPN0eUm', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function')])
        llm_message = response.choices[0].message

        # now, we need to find a way to extract the details of the function it wants to run and run it.
        tool_response = handle_tool_call(llm_message) #this will return the response to us in tool format

        # and now, we simply append the information we receive after running it, also append the message that we had received from the LLM to messages
        messages.append(llm_message) #type: ignore
        messages.append(tool_response)
        # and now we send the info to the LLM to process
        response = openai_client.chat.completions.create(model='gpt-4.1-mini-2025-04-14', messages=messages) #type: ignore

    # now, we return the response from the LLM, also, this is outside the if, so that in both cases, whether if runs or not, we return the respone
    return response.choices[0].message.content

In [25]:
# now we create the chat interface and run this
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


above we get the info that the tool has been called for london and few other place, and we also get the price in the chat. our tool worked nicely here

In [32]:
# now, as we can see there is a slight problem in our code. If we ask the system price for two locations in the same prompt, it will give us an error. Let's first understand why this happens and then understanding and implementing the solution will become very easy.

# The problem is in the handle_tool_call function. under the definition, we first assign message.tool_calls[0] to variable tool_call. Now, by doing this, the value that gets assigned to tool_call is simply the first function mentioned in the tool_calls. What if we have more than one function? Therefore, to check the presence of more than one function in the tool call, we need to iterate over it. So, that we get all the functions and then, we can check if the tool is being called for more than one function. Here's the example of the refined function, which we are now calling handle_tool_calls.

# Also, since we we can have more than one response (if the tool is called more than once), we create a list of responses, and append the responses with the formatted output for LLM.

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == 'get_ticket_price':
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            price_details = get_ticket_price(city)
            responses.append({
                'role': 'tool',
                'content': price_details,
                "tool_call_id": tool_call.id}
            )

    return responses

In [38]:
# lets now, run the chat interface and see what happens.

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai_client.chat.completions.create(model='gpt-4.1-mini', messages=messages, tools=tools) #type: ignore

    if response.choices[0].finish_reason == 'tool_calls':
        llm_message = response.choices[0].message
        tool_response = handle_tool_calls(llm_message) #added new function here
        messages.append(llm_message) #type: ignore
        # also, since we are using extend, so that the dicts inside the lists are added to the existing lists just like we are doing above for messages
        messages.extend(tool_response)
        response = openai_client.chat.completions.create(model='gpt-4.1-mini-2025-04-14', messages=messages) #type: ignore

    # now, we return the response from the LLM, also, this is outside the if, so that in both cases, whether if runs or not, we return the respone
    return response.choices[0].message.content

In [39]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Tool called for city: chennai
Tool called for city: mumbai
